# Notebook 11 — Strategy Backtesting

Convert Notebook 10 walk-forward predictions into explicit trading strategies.

**Inputs:** `data/interim/sp500_walk_forward_predictions.parquet`

**Strategies:** Buy & Hold, model Long/Cash, confidence-threshold Long/Cash.

**Evaluation:** cumulative return, CAGR, volatility, Sharpe, Sortino, maximum drawdown, Calmar, win rate, trades, turnover, transaction costs, slippage, annual returns, rolling Sharpe, and cost sensitivity.

Positions are generated from predictions available at the end of day *t* and applied to `next_day_return` from *t* to *t+1*.


## 1. Imports and configuration

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    for c in [Path.cwd(), Path.cwd().parent, Path("/mnt/data/quant-trading-research")]:
        if (c / "data").exists() and (c / "notebooks").exists():
            ROOT = c
            break

WF_PATH = ROOT / "data" / "interim" / "sp500_walk_forward_predictions.parquet"
INTERIM = ROOT / "data" / "interim"
FIGURES = ROOT / "reports" / "figures"
TABLES = ROOT / "reports" / "tables"
REPORTS = ROOT / "reports" / "generated"

for x in [INTERIM, FIGURES, TABLES, REPORTS]:
    x.mkdir(parents=True, exist_ok=True)

TRADING_DAYS = 252
TRANSACTION_COST_BPS = 5.0
SLIPPAGE_BPS = 2.0
TOTAL_COST_RATE = (TRANSACTION_COST_BPS + SLIPPAGE_BPS) / 10000
DEFAULT_THRESHOLD = 0.50
HIGH_CONFIDENCE_THRESHOLD = 0.60

print("Walk-forward file:", WF_PATH)
print("Total turnover cost:", TRANSACTION_COST_BPS + SLIPPAGE_BPS, "bps")


## 2. Load and validate walk-forward predictions

In [ ]:
if not WF_PATH.exists():
    raise FileNotFoundError(
        f"{WF_PATH} not found. Run Notebook 10 first."
    )

wf = pd.read_parquet(WF_PATH)
required = ["Date","Close","next_day_return","target","model","probability_up","prediction","fold"]
missing = [c for c in required if c not in wf.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

wf["Date"] = pd.to_datetime(wf["Date"], errors="coerce")
wf = wf.sort_values(["model","Date"]).reset_index(drop=True)

if wf["Date"].isna().any():
    raise ValueError("Invalid dates found.")
if wf["probability_up"].between(0,1).all() is False:
    raise ValueError("Invalid probabilities found.")

for m in wf["model"].unique():
    s = wf[wf["model"] == m]
    assert s["Date"].is_unique

print("Rows:", len(wf))
print("Models:", sorted(wf["model"].unique()))
print("Range:", wf["Date"].min().date(), "to", wf["Date"].max().date())


## 3. Performance metric functions

In [ ]:
def drawdown(equity):
    return equity / equity.cummax() - 1

def annualized_return(equity):
    if len(equity) < 2 or equity.iloc[-1] <= 0:
        return np.nan
    years = len(equity) / TRADING_DAYS
    return equity.iloc[-1] ** (1 / years) - 1

def sharpe(r):
    r = pd.Series(r).dropna()
    s = r.std(ddof=1)
    return np.nan if s == 0 else r.mean() / s * np.sqrt(TRADING_DAYS)

def sortino(r):
    r = pd.Series(r).dropna()
    downside = r[r < 0]
    if len(downside) == 0:
        return np.inf
    d = np.sqrt(np.mean(downside**2))
    return np.nan if d == 0 else r.mean() / d * np.sqrt(TRADING_DAYS)

def summarize(r, position, name):
    r = pd.Series(r).fillna(0)
    position = pd.Series(position, index=r.index).fillna(0)
    equity = (1 + r).cumprod()
    dd = drawdown(equity)
    turnover = position.diff().abs().fillna(position.abs())
    active = position != 0
    cagr = annualized_return(equity)
    mdd = dd.min()
    return {
        "strategy": name,
        "cumulative_return": equity.iloc[-1] - 1,
        "CAGR": cagr,
        "annualized_volatility": r.std(ddof=1) * np.sqrt(TRADING_DAYS),
        "Sharpe": sharpe(r),
        "Sortino": sortino(r),
        "max_drawdown": mdd,
        "Calmar": cagr / abs(mdd) if mdd < 0 else np.nan,
        "win_rate": (r[active] > 0).mean() if active.any() else np.nan,
        "active_days": int(active.sum()),
        "trades": int((turnover > 0).sum()),
        "turnover": turnover.sum(),
        "average_position": position.mean(),
    }, equity, dd, turnover

def backtest(s, threshold=0.5, cost_rate=TOTAL_COST_RATE, name="Strategy"):
    s = s.sort_values("Date").reset_index(drop=True).copy()
    position = (s["probability_up"] >= threshold).astype(float)
    gross = position * s["next_day_return"]
    turnover = position.diff().abs().fillna(position.abs())
    costs = turnover * cost_rate
    net = gross - costs
    out = s[["Date","Close","next_day_return","probability_up","target","fold"]].copy()
    out["position"] = position
    out["turnover"] = turnover
    out["transaction_cost"] = costs
    out["gross_return"] = gross
    out["net_return"] = net
    out["equity"] = (1 + net).cumprod()
    summary, equity, dd, turn = summarize(net, position, name)
    return out, summary, equity, dd, turn


## 4. Buy-and-hold benchmark

In [ ]:
benchmark_rows = []
benchmark_curves = {}
benchmark_frames = {}

for model in sorted(wf["model"].unique()):
    s = wf[wf["model"] == model].sort_values("Date").reset_index(drop=True)
    r = s["next_day_return"].fillna(0)
    pos = pd.Series(1.0, index=r.index)
    summary, equity, dd, turn = summarize(r, pos, "Buy & Hold")
    benchmark_rows.append(summary)
    benchmark_curves[model] = equity
    benchmark_frames[model] = s

benchmark = pd.DataFrame(benchmark_rows).iloc[0].to_dict()
print(pd.DataFrame([benchmark]))


## 5. Backtest all model Long/Cash strategies

In [ ]:
strategy_frames = {}
strategy_curves = {}
strategy_drawdowns = {}
results = []

for model in sorted(wf["model"].unique()):
    s = wf[wf["model"] == model].sort_values("Date")
    frame, summary, equity, dd, turn = backtest(
        s, DEFAULT_THRESHOLD, TOTAL_COST_RATE,
        f"{model} Long/Cash"
    )
    strategy_frames[summary["strategy"]] = frame
    strategy_curves[summary["strategy"]] = equity
    strategy_drawdowns[summary["strategy"]] = dd
    results.append(summary)

strategy_df = pd.DataFrame(results).sort_values(
    "Sharpe", ascending=False, na_position="last"
).reset_index(drop=True)

comparison = pd.concat(
    [pd.DataFrame([benchmark]), strategy_df],
    ignore_index=True
).sort_values("Sharpe", ascending=False, na_position="last")

display(comparison)
comparison.to_csv(TABLES / "sp500_strategy_backtest_comparison.csv", index=False)


## 6. Equity curves

In [ ]:
fig = plt.figure(figsize=(15,7))
for model, eq in benchmark_curves.items():
    plt.plot(benchmark_frames[model]["Date"], eq, label="Buy & Hold")
    break
for name, eq in strategy_curves.items():
    plt.plot(strategy_frames[name]["Date"], eq, label=name)
plt.title("Walk-Forward Strategy Equity Curves")
plt.xlabel("Date"); plt.ylabel("Equity"); plt.legend(); plt.grid(True, alpha=.25)
plt.tight_layout()
path = FIGURES / "sp500_strategy_equity_curves.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(path)


## 7. Drawdown curves

In [ ]:
fig = plt.figure(figsize=(15,7))
for model, eq in benchmark_curves.items():
    dates = benchmark_frames[model]["Date"]
    plt.plot(dates, drawdown(eq), label="Buy & Hold")
    break
for name, dd in strategy_drawdowns.items():
    plt.plot(strategy_frames[name]["Date"], dd, label=name)
plt.title("Walk-Forward Strategy Drawdowns")
plt.xlabel("Date"); plt.ylabel("Drawdown"); plt.legend(); plt.grid(True, alpha=.25)
plt.tight_layout()
path = FIGURES / "sp500_strategy_drawdowns.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(path)


## 8. Select the best model-based strategy by Sharpe

In [ ]:
best_row = strategy_df.iloc[0]
best_name = best_row["strategy"]
best_frame = strategy_frames[best_name]
print("Best model-based strategy:", best_name)
display(pd.DataFrame([best_row]))


## 9. Confidence-threshold strategies

In [ ]:
high_results = []
high_frames = {}

for model in sorted(wf["model"].unique()):
    s = wf[wf["model"] == model].sort_values("Date")
    frame, summary, equity, dd, turn = backtest(
        s, HIGH_CONFIDENCE_THRESHOLD, TOTAL_COST_RATE,
        f"{model} Threshold {HIGH_CONFIDENCE_THRESHOLD:.2f}"
    )
    high_results.append(summary)
    high_frames[summary["strategy"]] = frame

high_df = pd.DataFrame(high_results).sort_values(
    "Sharpe", ascending=False, na_position="last"
)
display(high_df)
high_df.to_csv(TABLES / "sp500_high_confidence_strategy_comparison.csv", index=False)


## 10. Transaction-cost sensitivity

In [ ]:
rows = []
for bps in [0, 2, 5, 10, 20]:
    rate = bps / 10000
    for model in sorted(wf["model"].unique()):
        s = wf[wf["model"] == model].sort_values("Date")
        _, summary, _, _, _ = backtest(
            s, DEFAULT_THRESHOLD, rate,
            f"{model} @ {bps} bps"
        )
        rows.append({
            "model": model,
            "cost_bps": bps,
            "CAGR": summary["CAGR"],
            "Sharpe": summary["Sharpe"],
            "Sortino": summary["Sortino"],
            "max_drawdown": summary["max_drawdown"],
            "cumulative_return": summary["cumulative_return"],
            "trades": summary["trades"],
            "turnover": summary["turnover"],
        })
cost_df = pd.DataFrame(rows)
display(cost_df)
cost_df.to_csv(TABLES / "sp500_strategy_cost_sensitivity.csv", index=False)


## 11. Cost sensitivity plot

In [ ]:
fig = plt.figure(figsize=(14,6))
for model in cost_df["model"].unique():
    s = cost_df[cost_df["model"] == model]
    plt.plot(s["cost_bps"], s["Sharpe"], marker="o", label=model)
plt.axhline(0, linestyle="--")
plt.title("Sharpe Sensitivity to Trading Costs")
plt.xlabel("Total Cost (bps)"); plt.ylabel("Sharpe")
plt.legend(); plt.grid(True, alpha=.25); plt.tight_layout()
path = FIGURES / "sp500_strategy_cost_sensitivity.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(path)


## 12. Gross vs net performance

In [ ]:
rows = []
for model in sorted(wf["model"].unique()):
    s = wf[wf["model"] == model].sort_values("Date").reset_index(drop=True)
    pos = (s["probability_up"] >= DEFAULT_THRESHOLD).astype(float)
    gross = pos * s["next_day_return"]
    turnover = pos.diff().abs().fillna(pos.abs())
    net = gross - turnover * TOTAL_COST_RATE
    gs, *_ = summarize(gross, pos, f"{model} Gross")
    ns, *_ = summarize(net, pos, f"{model} Net")
    rows.extend([gs, ns])
gross_net = pd.DataFrame(rows)
display(gross_net)
gross_net.to_csv(TABLES / "sp500_strategy_gross_vs_net.csv", index=False)


## 13. Position history for best strategy

In [ ]:
fig = plt.figure(figsize=(15,5))
plt.step(best_frame["Date"], best_frame["position"], where="post")
plt.title(f"Position History — {best_name}")
plt.xlabel("Date"); plt.ylabel("Position"); plt.grid(True, alpha=.25)
plt.tight_layout()
path = FIGURES / "sp500_best_strategy_positions.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(path)


## 14. Rolling 252-day Sharpe

In [ ]:
r = best_frame.set_index("Date")["net_return"]
rolling = r.rolling(TRADING_DAYS).apply(
    lambda x: x.mean()/x.std(ddof=1)*np.sqrt(TRADING_DAYS)
    if x.std(ddof=1) > 0 else np.nan
)
fig = plt.figure(figsize=(15,5))
plt.plot(rolling.index, rolling)
plt.axhline(0, linestyle="--")
plt.title(f"Rolling 252-Day Sharpe — {best_name}")
plt.xlabel("Date"); plt.ylabel("Sharpe"); plt.grid(True, alpha=.25)
plt.tight_layout()
path = FIGURES / "sp500_best_strategy_rolling_sharpe.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(path)


## 15. Annual returns

In [ ]:
annual = []
for name, frame in strategy_frames.items():
    x = frame[["Date","net_return"]].copy()
    x["year"] = x["Date"].dt.year
    y = x.groupby("year")["net_return"].apply(lambda z: (1+z).prod()-1).reset_index()
    y["strategy"] = name
    annual.append(y)

b = benchmark_frames[next(iter(benchmark_frames))]
x = b[["Date","next_day_return"]].copy()
x["year"] = x["Date"].dt.year
y = x.groupby("year")["next_day_return"].apply(lambda z: (1+z).prod()-1).reset_index()
y = y.rename(columns={"next_day_return":"net_return"})
y["strategy"] = "Buy & Hold"
annual_df = pd.concat(annual + [y], ignore_index=True)
display(annual_df.tail(30))
annual_df.to_csv(TABLES / "sp500_strategy_annual_returns.csv", index=False)


## 16. Best strategy annual return chart

In [ ]:
x = annual_df[annual_df["strategy"] == best_name]
fig = plt.figure(figsize=(14,6))
plt.bar(x["year"].astype(str), x["net_return"])
plt.axhline(0, linestyle="--")
plt.title(f"Annual Returns — {best_name}")
plt.xlabel("Year"); plt.ylabel("Return"); plt.xticks(rotation=45)
plt.grid(True, axis="y", alpha=.25); plt.tight_layout()
path = FIGURES / "sp500_best_strategy_annual_returns.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(path)


## 17. Monthly returns

In [ ]:
monthly = []
for name, frame in strategy_frames.items():
    x = frame[["Date","net_return"]].copy()
    x["month"] = x["Date"].dt.to_period("M").dt.to_timestamp()
    y = x.groupby("month")["net_return"].apply(lambda z: (1+z).prod()-1).reset_index()
    y["strategy"] = name
    monthly.append(y)

x = b[["Date","next_day_return"]].copy()
x["month"] = x["Date"].dt.to_period("M").dt.to_timestamp()
y = x.groupby("month")["next_day_return"].apply(lambda z: (1+z).prod()-1).reset_index()
y = y.rename(columns={"next_day_return":"net_return"})
y["strategy"] = "Buy & Hold"
monthly_df = pd.concat(monthly + [y], ignore_index=True)
monthly_df.to_csv(TABLES / "sp500_strategy_monthly_returns.csv", index=False)
display(monthly_df.head(20))


## 18. Save best strategy backtest

In [ ]:
best_path = INTERIM / "sp500_best_strategy_backtest.parquet"
best_frame.to_parquet(best_path, index=False)
print("Saved:", best_path)


## 19. Final strategy comparison

In [ ]:
risk_summary = comparison[
    ["strategy","CAGR","annualized_volatility","Sharpe","Sortino",
     "max_drawdown","Calmar","win_rate","trades","turnover"]
].sort_values("Sharpe", ascending=False, na_position="last")
display(risk_summary)


## 20. Backtest report

In [ ]:
report = {
    "evaluation_range": {
        "start": wf["Date"].min().strftime("%Y-%m-%d"),
        "end": wf["Date"].max().strftime("%Y-%m-%d")
    },
    "cost_assumptions": {
        "transaction_cost_bps": TRANSACTION_COST_BPS,
        "slippage_bps": SLIPPAGE_BPS,
        "total_cost_bps": TRANSACTION_COST_BPS + SLIPPAGE_BPS
    },
    "strategies": comparison.to_dict(orient="records"),
    "high_confidence_strategies": high_df.to_dict(orient="records"),
    "best_model_strategy": best_name,
    "cost_sensitivity": cost_df.to_dict(orient="records"),
    "methodological_note": (
        "Signals are generated from walk-forward predictions and applied "
        "to next-day returns. Costs are charged on position changes. "
        "Results are historical research results, not guarantees of future performance."
    )
}
report_path = REPORTS / "sp500_strategy_backtest_report.json"
report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print(json.dumps(report, indent=2, default=str))
print("Saved:", report_path)


## 21. Raw master dataset integrity check

In [ ]:
MASTER_PATH = ROOT / "data" / "raw" / "sp500_1950_present.csv"
master = pd.read_csv(MASTER_PATH, low_memory=False)
expected = ["Date","Open","High","Low","Close","Adj.Close","Volume"]
assert list(master.columns) == expected
dates = pd.to_datetime(master["Date"], errors="coerce")
assert dates.notna().all()
assert dates.is_unique
assert dates.is_monotonic_increasing
print("Raw master dataset integrity after strategy backtesting: PASS")
print("Rows:", len(master))


# Notebook 11 Complete

Notebook 11 converts pseudo-out-of-sample predictions into explicit trading strategies and evaluates realistic risk/return behavior.

**Next:** Notebook 12 — Robustness, Stress Testing & Final Model Freeze.

Run this notebook completely and verify the results before continuing.
